# 2.1 RF Baseline LOLO

In [1]:
# 01_rf_baseline_lolo.ipynb
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from common.training import run_lolo_evaluation, format_summary

# Load feature table đã lưu ở GĐ1
feature_df = pd.read_csv("outputs/tables/feature_table_full.csv")

# Tách tên các cột đặc trưng (bỏ cột metadata)
feature_cols = [c for c in feature_df.columns 
                if c.startswith(("time_", "order_", "envelope_"))]
print(f"Số đặc trưng: {len(feature_cols)}")
print(f"Số file: {len(feature_df)}")
print(f"Phân phối label:\n{feature_df['label'].value_counts()}")

# Chạy RF qua 4 fold LOLO
per_fold, summary = run_lolo_evaluation(
    feature_df=feature_df,
    feature_cols=feature_cols,
    estimator_factory=lambda: RandomForestClassifier(
        n_estimators=300, random_state=42, n_jobs=-1
    ),
)

print("\n=== Per-fold ===")
print(per_fold.to_string())
print(f"\nAccuracy LOLO: {format_summary(summary, 'accuracy')}")
print(f"F1 LOLO:       {format_summary(summary, 'f1_macro')}")

Số đặc trưng: 32
Số file: 40
Phân phối label:
label
B         12
IR        12
OR        12
Normal     4
Name: count, dtype: int64

=== Per-fold ===
     fold_name  test_load trainval_loads  n_train  n_val  n_test  accuracy  f1_macro
0  test_load_0          0      [1, 2, 3]       24      6      10       1.0       1.0
1  test_load_1          1      [0, 2, 3]       24      6      10       1.0       1.0
2  test_load_2          2      [0, 1, 3]       24      6      10       1.0       1.0
3  test_load_3          3      [0, 1, 2]       24      6      10       1.0       1.0

Accuracy LOLO: 1.0000 ± 0.0000
F1 LOLO:       1.0000 ± 0.0000


# 2.2 Feature Selection

# 2.3 MLP LOLO

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping
from common.models import build_mlp, compile_classifier
from common.training import run_lolo_evaluation, format_summary

feature_df = pd.read_csv("outputs/tables/feature_table_full.csv")
feature_cols = [c for c in feature_df.columns 
                if c.startswith(("time_", "order_", "envelope_"))]
for c in feature_cols:
    feature_df[c] = pd.to_numeric(feature_df[c], errors='coerce')

class KerasMLPWrapper:
    def __init__(self, input_dim):
        self.input_dim = input_dim
        # Khởi tạo ngay từ đầu thay vì để None để VS Code không báo lỗi
        self.scaler = StandardScaler()
        self.label_enc = LabelEncoder()
        self.model = build_mlp(input_dim=self.input_dim)
        compile_classifier(self.model, learning_rate=1e-3)
        
    def fit(self, X, y):
        X_arr = np.asarray(X, dtype=np.float64)
        X_s = self.scaler.fit_transform(X_arr)
        
        # Ép kiểu y_enc về np.int32 để xóa lỗi gạch đỏ "ArrayLike"
        y_enc = np.asarray(self.label_enc.fit_transform(y), dtype=np.int32)
        
        self.model.fit(
            X_s, y_enc,
            epochs=500,
            batch_size=4,
            verbose=0,
            callbacks=[EarlyStopping(monitor='loss', patience=50, 
                                     restore_best_weights=True)],
        )
        return self
        
    def predict(self, X):
        X_arr = np.asarray(X, dtype=np.float64)
        X_s = self.scaler.transform(X_arr)
        probs = self.model.predict(X_s, verbose=0)
        return self.label_enc.inverse_transform(np.argmax(probs, axis=1))

# Full 32
per_fold_32, summary_32 = run_lolo_evaluation(
    feature_df=feature_df, feature_cols=feature_cols,
    estimator_factory=lambda: KerasMLPWrapper(input_dim=32),
)
print("=== MLP (32 đặc trưng) ===")
print(per_fold_32.to_string())
print(f"Accuracy: {format_summary(summary_32, 'accuracy')}")
print(f"F1:       {format_summary(summary_32, 'f1_macro')}")

=== MLP (32 đặc trưng) ===
     fold_name  test_load trainval_loads  n_train  n_val  n_test  accuracy  f1_macro
0  test_load_0          0      [1, 2, 3]       24      6      10       0.8    0.8375
1  test_load_1          1      [0, 2, 3]       24      6      10       1.0    1.0000
2  test_load_2          2      [0, 1, 3]       24      6      10       1.0    1.0000
3  test_load_3          3      [0, 1, 2]       24      6      10       1.0    1.0000
Accuracy: 0.9500 ± 0.1000
F1:       0.9594 ± 0.0813


In [ ]:
# Xem chi tiết fold test_load_0
from common.training import iterate_lolo_splits
import numpy as np

for fold_info, train_df, val_df, test_df in iterate_lolo_splits(feature_df):
    if fold_info["test_load"] == 0:
        break

# Train model trên fold này
wrapper = KerasMLPWrapper(input_dim=32)
wrapper.fit(train_df[feature_cols], train_df["label"])
y_pred = wrapper.predict(test_df[feature_cols])
y_true = test_df["label"].values

# In từng file đúng/sai
print("file_id | thật | dự đoán | đúng?")
for fid, t, p in zip(test_df["file_id"], y_true, y_pred):
    mark = "✅" if t == p else "❌ SAI"
    print(f"{fid.split('/')[-1]:<20} {t:<7} {p:<7} {mark}")

# 2.4 INT8